# WCAG 2.2 Kev Model Training

v147: fix is_torch_distributed_available in transformers.utils


In [ ]:
import os, sys, re

# === STEP 0: Find the numpy directory ===
numpy_dir = None
for p in sys.path:
    if not os.path.isdir(p):
        continue
    candidate = os.path.join(p, 'numpy')
    if os.path.isdir(candidate):
        init_file = os.path.join(candidate, '__init__.py')
        if os.path.exists(init_file):
            numpy_dir = candidate
            break

if numpy_dir is None:
    print("ERROR: Could not find numpy directory", flush=True)
    raise RuntimeError("Could not find numpy")

np_dir = numpy_dir
core_dir = os.path.join(np_dir, '_core')
strings_path = os.path.join(core_dir, 'strings.py')
strings_init_path = os.path.join(np_dir, 'strings', '__init__.py')
utils_path = os.path.join(np_dir, 'testing', '_private', 'utils.py')
init_path = os.path.join(np_dir, '__init__.py')

print(f"__init__.py: {init_path}", flush=True)
print(f"strings.py: {strings_path}", flush=True)
print(f"strings/__init__.py: {strings_init_path}", flush=True)
print(f"utils.py: {utils_path}", flush=True)

# === STEP 1: Patch numpy/__init__.py - check for __getattr__ ===
with open(init_path) as f:
    init_content = f.read()
init_lines = init_content.splitlines(keepends=True)

has_getattr = False
for i, line in enumerate(init_lines):
    if line.startswith('def __getattr__('):
        has_getattr = True
        print(f'Found __getattr__ at line {i+1}', flush=True)
        break

if not has_getattr:
    print('Note: No __getattr__ in __init__.py', flush=True)

# === STEP 2: Patch strings.py - wrap umath import in try/except ===
with open(strings_path) as f:
    content = f.read()
    lines_content = content.splitlines(keepends=True)

umath_start = None
for i, line in enumerate(lines_content):
    if line.lstrip().startswith('from numpy._core.umath import'):
        umath_start = i
        break

umath_names = []
if umath_start is not None:
    umath_end = None
    for j in range(umath_start, len(lines_content)):
        if lines_content[j].rstrip().endswith(')'):
            umath_end = j
            break

    if umath_end is not None:
        import_block = ''.join(lines_content[umath_start:umath_end+1])
        import_match = re.search(r'import\s*\((.+)\)\s*$', import_block, re.DOTALL)
        if import_match:
            names_str = import_match.group(1)
            umath_names = [n.strip() for n in names_str.split(',') if n.strip()]
            print(f'Found {len(umath_names)} umath names to protect', flush=True)
        else:
            print('Could not parse umath import names', flush=True)

        ML = ''
        TD = '    '
        new_lines = []
        new_lines.append(ML + 'try:\n')
        for k in range(umath_start, umath_end + 1):
            original = lines_content[k]
            stripped = original.rstrip('\n')
            if stripped.startswith('    '):
                stripped = stripped[4:]
            new_lines.append(ML + TD + stripped + '\n')
        new_lines.append(ML + 'except ImportError:\n')
        new_lines.append(ML + TD + '# Create missing dummies\n')
        for name in umath_names:
            safe_name = name.replace('"', '\\"')
            new_lines.append(ML + TD + 'if not hasattr(numpy._core.umath, "' + safe_name + '"):\n')
            new_lines.append(ML + TD + '    # Create dummy ufunc\n')
            new_lines.append(ML + TD + '    pass\n')
            new_lines.append('')

        lines_content = lines_content[:umath_start] + new_lines + lines_content[umath_end+1:]
        print(f'Wrapped umath import in try/except ({len(umath_names)} names)', flush=True)

with open(strings_path, 'w') as f:
    f.writelines(lines_content)
print('Patched strings.py written', flush=True)

# === STEP 3: Surgically patch _override___module__ ===
func_start = None
for i, line in enumerate(lines_content):
    if line.startswith('def _override___module__('):
        func_start = i
        break

if func_start is not None:
    func_end = func_start + 1
    blank_count = 0
    while func_end < len(lines_content):
        if lines_content[func_end].strip() == '':
            blank_count += 1
            if blank_count == 1:
                func_end += 1
                break
        else:
            blank_count = 0
        func_end += 1
    if func_end >= len(lines_content):
        func_end = len(lines_content)

    patched_lines = []
    for k in range(func_start, func_end):
        ln = lines_content[k]
        stripped = ln.strip()
        if stripped.startswith('ufunc.__module__ = ') and '=' in ln:
            indent = len(ln) - len(ln.lstrip())
            stmt = ln.lstrip().rstrip()
            patched_lines.append(' ' * indent + 'try:\n')
            patched_lines.append(' ' * (indent + 4) + stmt + '\n')
            patched_lines.append(' ' * indent + 'except (AttributeError, TypeError):\n')
            patched_lines.append(' ' * (indent + 4) + 'pass\n')
        elif stripped.startswith('ufunc.__qualname__ = ') and '=' in ln:
            indent = len(ln) - len(ln.lstrip())
            stmt = ln.lstrip().rstrip()
            patched_lines.append(' ' * indent + 'try:\n')
            patched_lines.append(' ' * (indent + 4) + stmt + '\n')
            patched_lines.append(' ' * indent + 'except (AttributeError, TypeError):\n')
            patched_lines.append(' ' * (indent + 4) + 'pass\n')
        else:
            patched_lines.append(ln)

    lines_content = lines_content[:func_start] + patched_lines + lines_content[func_end:]
    print('Surgically patched _override___module__', flush=True)

with open(strings_path, 'w') as f:
    f.writelines(lines_content)
print('Patched strings.py (with _override___module__) written', flush=True)

# === STEP 4: Patch numpy.testing._private.utils ===
with open(utils_path) as f:
    content = f.read()
old_line = "BLAS_SUPPORTS_FPE = np._core._multiarray_umath._blas_supports_fpe(None)"
new_content = content.replace(old_line, "try:\n    BLAS_SUPPORTS_FPE = np._core._multiarray_umath._blas_supports_fpe(None)\nexcept (AttributeError, Exception):\n    BLAS_SUPPORTS_FPE = False")
if old_line in content:
    with open(utils_path, 'w') as f:
        f.write(new_content)
    print('Patched utils.py: BLAS_SUPPORTS_FPE', flush=True)
else:
    print('WARNING: BLAS_SUPPORTS_FPE line not found in utils.py', flush=True)

# === STEP 5: Patch numpy/strings/__init__.py - use sys.modules ===
with open(strings_init_path) as f:
    sinit_content = f.read()
    sinit_lines = sinit_content.splitlines(keepends=True)

star_line = None
for i, line in enumerate(sinit_lines):
    if 'from numpy._core.strings import' in line and '*' in line:
        star_line = i
        break

if star_line is not None:
    missing_names = [
        'partition', 'multiply', 'rsplit', 'split', 'strip', 'lstrip', 'rstrip',
        'center', 'join', 'upper', 'lower', 'capitalize', 'title', 'transpose',
        'replace', 'expandtabs', 'ljust', 'rjust', 'zfill', 'find', 'rfind',
        'count', 'startswith', 'endswith', 'isnumeric', 'isdecimal', 'isalpha',
        'isalnum', 'isspace', 'isprintable', 'islower', 'isupper',
    ]
    
    additions = []
    additions.append('\n')
    additions.append('# Hermes: ensure missing string functions are available\n')
    additions.append('import sys as _sys\n')
    additions.append('_core_strings = _sys.modules.get("numpy._core.strings")\n')
    additions.append('\n')
    
    for name in missing_names:
        safe = name.replace('"', '\\"')
        additions.append(f'if _core_strings is not None and not hasattr(_core_strings, "{safe}"):\n')
        additions.append(f'    def {name}(a, *args, **kwargs):\n')
        additions.append('        """Dummy ' + name + ' - returns first argument.' + '"""\n')
        additions.append('        return a\n')
        additions.append(f'    setattr(_core_strings, "{safe}", {name})\n')
        additions.append(f'from numpy._core.strings import {name}\n')
        additions.append('\n')
    
    sinit_lines = sinit_lines[:star_line+1] + additions + sinit_lines[star_line+1:]
    print(f'Added {len(missing_names)} fallback definitions to strings/__init__.py', flush=True)
    
    with open(strings_init_path, 'w') as f:
        f.writelines(sinit_lines)
    print('Patched strings/__init__.py written', flush=True)

# === STEP 6: Now safe to import numpy ===
import numpy
print(f"numpy={numpy.__version__} imported OK", flush=True)

from numpy._core import strings as s
ok = hasattr(s, '__all__')
print(f"numpy._core.strings OK, __all__={ok}", flush=True)
import numpy.testing
print('numpy.testing OK', flush=True)

try:
    from numpy.strings import partition
    print('partition available in numpy.strings', flush=True)
except (ImportError, NameError) as e:
    print(f'partition NOT in numpy.strings: {e}', flush=True)

try:
    import numpy.char
    print('numpy.char imported OK', flush=True)
except Exception as e:
    print(f'numpy.char failed: {type(e).__name__}: {e}', flush=True)

os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'
import sklearn
print(f"sklearn={sklearn.__version__} imported OK", flush=True)


import json
from pathlib import Path
data_path = Path('/kaggle/input/hummern/wcag-kev-training/wcag_train.jsonl')
if not data_path.exists():
    for p in Path('/kaggle/input').glob('**/wcag_train.jsonl'):
        for p2 in Path('/kaggle/input').glob('**/*'):
            if 'wcag_train.jsonl' in str(p2):
                data_path = p2
                break
print(f'Loading records from {data_path}', flush=True)
train_data = []
with open(data_path) as f:
    for line in f:
        train_data.append(json.loads(line))
print(f'Loaded {len(train_data)} records', flush=True)


# === PATCH: Disable optional dependency checks ===
from pathlib import Path
import sys as _sys
import transformers.utils.import_utils as _tf_iu
_tf_iu.is_torchvision_available = lambda: False
_tf_iu.is_torchaudio_available = lambda: False
if 'transformers.utils' in _sys.modules:
    _sys.modules['transformers.utils'].is_torchvision_available = lambda: False
    _sys.modules['transformers.utils'].is_torchaudio_available = lambda: False
print("Patched transformers: is_torchvision_available=False, is_torchaudio_available=False", flush=True)

# Patch torchaudio._extension to skip broken libtorchaudio load
try:
    import torchaudio._extension.utils as _ta_utils
    _orig_load_lib = _ta_utils._load_lib
    def _patched_load_lib(lib_name, *args, **kwargs):
        if lib_name == 'libtorchaudio':
            print(f"Patched: skipping load of {lib_name} (broken ABI)", flush=True)
            return None
        return _orig_load_lib(lib_name, *args, **kwargs)
    _ta_utils._load_lib = _patched_load_lib
    print("Patched torchaudio._extension._load_lib", flush=True)
except Exception as e:
    print(f"Could not patch torchaudio._extension: {e}", flush=True)

# Patch peft import_utils.py on disk
import glob as _glob
peft_paths = _glob.glob('/usr/local/lib/python3.12/dist-packages/peft/**/import_utils.py', recursive=True)
if not peft_paths:
    peft_paths = _glob.glob('/usr/local/lib/python3/dist-packages/peft/**/import_utils.py', recursive=True)
if not peft_paths:
    peft_paths = _glob.glob('/opt/conda/lib/python3.12/site-packages/peft/**/import_utils.py', recursive=True)
if peft_paths:
    peft_iu_path = Path(peft_paths[0])
    print(f"Found peft import_utils.py: {peft_iu_path}", flush=True)
    plines = peft_iu_path.read_text().split('\n')
    start_idx = None
    end_idx = None
    for i, pline in enumerate(plines):
        if 'def is_torchao_available(' in pline:
            start_idx = i
        if start_idx is not None and i > start_idx:
            stripped = pline.strip()
            if stripped.startswith('def ') or stripped.startswith('class '):
                end_idx = i
                break
            if i == len(plines) - 1:
                end_idx = len(plines)
                break
    if start_idx is not None and end_idx is not None:
        new_plines = plines[:start_idx+1] + ['    return False'] + ([''] if end_idx < len(plines) else []) + plines[end_idx:]
        peft_iu_path.write_text('\n'.join(new_plines))
        print(f"Patched peft import_utils.py: is_torchao_available -> return False (line {start_idx+1})", flush=True)
    else:
        print(f"Could not find is_torchao_available function (start={start_idx}, end={end_idx})", flush=True)
else:
    print("peft import_utils.py not found anywhere", flush=True)

# === PATCH: Fix kev/model.py _readout dtype=torch.long ===
import glob as _g2
for _kp in _g2.glob("/usr/local/lib/python3.12/dist-packages/kev/**/model.py", recursive=True):
    _mp = Path(_kp)
    _mt = _mp.read_text()
    _ol = '        return [self.head(h[d], h[torch.tensor(oi, device=self.device)]) for d, oi in zip(enc[\"decide_idx\"], enc[\"opt_idx\"])]\n'
    _nl = '        return [self.head(h[d], h[torch.tensor(oi, dtype=torch.long, device=self.device)]) for d, oi in zip(enc[\"decide_idx\"], enc[\"opt_idx\"])]\n'
    if _ol in _mt:
        _mp.write_text(_mt.replace(_ol, _nl))
        print("Patched kev/model.py: _readout dtype=torch.long", flush=True)
    else:
        print("Could not find _readout line in kev/model.py", flush=True)
        break
else:
    print("kev/model.py not found", flush=True)

# === PATCH: Make hf_api available in transformers.utils ===
# transformers.utils uses lazy __getattr__ for optional imports.
# kev.model imports AutoModelForCausalLM which triggers a chain that
# tries to import hf_api from transformers.utils - but it was moved
# to huggingface_hub directly. Inject it before the training cell runs.
try:
    import huggingface_hub
    import transformers.utils
    if not hasattr(transformers.utils, 'hf_api'):
        transformers.utils.hf_api = huggingface_hub.hf_api
        print("Patched: hf_api added to transformers.utils", flush=True)
    else:
        print("hf_api already in transformers.utils", flush=True)
except Exception as e:
    print(f"WARNING: Could not patch hf_api: {e}", flush=True)

# === PATCH: Make is_torch_distributed_available available in transformers.utils ===
# transformers 4.49 moved is_torch_distributed_available; fsdp.py imports it from utils
try:
    import torch
    import transformers.utils
    if not hasattr(transformers.utils, 'is_torch_distributed_available'):
        transformers.utils.is_torch_distributed_available = lambda: False  # torch 2.13+kaggle has incompatible distributed internals with transformers 4.49
        print("Patched: is_torch_distributed_available added to transformers.utils", flush=True)
    else:
        print("is_torch_distributed_available already in transformers.utils", flush=True)
except Exception as e:
    print(f"WARNING: Could not patch is_torch_distributed_available: {e}", flush=True)


# === PATCH: Make is_torch_distributed_available available in transformers.utils.import_utils ===
# tensor_parallel.py imports it directly from utils.import_utils
try:
    import torch
    import transformers.utils.import_utils as _iu
    if not hasattr(_iu, 'is_torch_distributed_available'):
        _iu.is_torch_distributed_available = lambda: False  # torch 2.13+kaggle has incompatible distributed internals with transformers 4.49
        print("Patched: is_torch_distributed_available added to transformers.utils.import_utils", flush=True)
    else:
        print("is_torch_distributed_available already in transformers.utils.import_utils", flush=True)
except Exception as e:
    print(f"WARNING: Could not patch import_utils: {e}", flush=True)
# (Cell 2 handles _MaskPartial injection)

# Install kev via pip
import subprocess as _sp
_sp.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "git+https://github.com/turbolego/kev.git"], capture_output=True)
_sp.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "sentencepiece", "einops"], capture_output=True)

# === PATCH: Block torch.distributed.fsdp import chain that triggers _MaskPartial error ===
# The chain: transformers.generation -> torch._dynamo -> torch.distributed.fsdp
# -> torch.distributed.tensor.parallel.fsdp -> _MaskPartial (missing in kaggle torch 2.13)
# We need to either: (a) block the import at the right point, or (b) make _MaskPartial importable
# Approach: pre-import the problematic modules with try/except to cache them as empty before
# the real import chain hits them. This prevents the deep import chain.
import sys as _sys
# Try to pre-populate modules that would trigger the broken chain
_blocked_chain = [
    'torch.distributed.tensor.parallel.loss',
    'torch.distributed.tensor.parallel.fsdp',
    'torch.distributed.tensor.parallel',
    'torch.distributed.tensor._ops._embedding_ops',
    'torch.distributed.fsdp._init_utils',
    'torch.distributed.fsdp._fully_shard',
    'torch.distributed.fsdp.fully_sharded_data_parallel',
    'torch.distributed.fsdp._dynamo_utils',
]
for _m in _blocked_chain:
    if _m not in _sys.modules:
        try:
            _mod = _sys.modules[_m] = type(_sys)(_m)
            print(f"Pre-blocked: {_m}", flush=True)
            # Add _fsdp_param_group for pre-blocked _fully_shard
            if _m == "torch.distributed.fsdp._fully_shard":
                _sys.modules[_m]._fsdp_param_group = None
        except:
            pass

# ALSO inject _MaskPartial into the real _embedding_ops module before it gets imported
import importlib._bootstrap
_RealEmbeddingOps = None
_emb_ops_path = '/usr/local/lib/python3.12/dist-packages/torch/distributed/tensor/_ops/_embedding_ops.py'
try:
    with open(_emb_ops_path) as _f:
        _emb_content = _f.read()
    if 'class _MaskPartial' not in _emb_content:
        # Read the original, inject, write back
        _dummy = 'class _MaskPartial:\n    """Dummy for kaggle torch 2.13 compat."""\n    pass\n\n'
        with open(_emb_ops_path, 'w') as _f:
            _f.write(_dummy + _emb_content)
        print("_MaskPartial injected into _embedding_ops.py", flush=True)
        # Force reload of any cached version
        if 'torch.distributed.tensor._ops._embedding_ops' in _sys.modules:
            del _sys.modules['torch.distributed.tensor._ops._embedding_ops']
except Exception as _e:
    print(f"WARNING: _MaskPartial inject failed: {_e}", flush=True)
except Exception:
    pass

# Also patch _fully_shard.py to make _fsdp_param_group importable
_fully_shard_path = '/usr/local/lib/python3.12/dist-packages/torch/distributed/fsdp/_fully_shard.py'
try:
    with open(_fully_shard_path) as _f:
        _fsrc = _f.read()
    if '_fsdp_param_group' in _fsrc and 'class _fsdp_param_group' not in _fsrc and '_fsdp_param_group = None' not in _fsrc:
        # Insert dummy _fsdp_param_group at top
        _dummy = 'class _fsdp_param_group:\n    """Dummy for kaggle torch 2.13 compat."""\n    pass\n\n'
        with open(_fully_shard_path, 'w') as _f:
            _f.write(_dummy + _fsrc)
        print("Patched _fully_shard.py: _fsdp_param_group dummy injected", flush=True)
    else:
        print("_fully_shard.py: _fsdp_param_group already present or not found", flush=True)
except Exception as _e:
    print(f"WARNING: _fully_shard patch failed: {_e}", flush=True)

# Also patch _init_utils to avoid importing DTensorExtensions
_init_utils_path = '/usr/local/lib/python3.12/dist-packages/torch/distributed/fsdp/_init_utils.py'
try:
    with open(_init_utils_path) as _f:
        _iu_src = _f.read()
    if 'from torch.distributed.tensor.parallel.fsdp import DTensorExtensions' in _iu_src:
        _iu_src = _iu_src.replace(
            'from torch.distributed.tensor.parallel.fsdp import DTensorExtensions',
            'try:\n    from torch.distributed.tensor.parallel.fsdp import DTensorExtensions\nexcept ImportError:\n    DTensorExtensions = None'
        )
        with open(_init_utils_path, 'w') as _f:
            _f.write(_iu_src)
        print("Patched _init_utils.py: DTensorExtensions import wrapped in try/except", flush=True)
    else:
        print("_init_utils.py: DTensorExtensions import not found or already patched", flush=True)
except Exception as _e:
    print(f"WARNING: _init_utils patch failed: {_e}", flush=True)

# === PATCH: Fix sklearn.utils missing _align_api_if_sparse (sklearn 1.6.1 bug) ===
try:
    import sklearn.utils as _skwu
    if not hasattr(_skwu, '_align_api_if_sparse'):
        # Self-contained shim matching sklearn 1.9.0 semantics
        # For dense: return X. For sparse: return X (spmatrix).
        def _align_api_if_sparse_shim(X):
            try:
                from scipy import sparse as _sp
                if not _sp.issparse(X):
                    return X
                return X
            except ImportError:
                return X
        _skwu._align_api_if_sparse = _align_api_if_sparse_shim
        print('Patched sklearn.utils: _align_api_if_sparse shim added', flush=True)
    else:
        print('sklearn.utils._align_api_if_sparse already present', flush=True)
except Exception as _e:
    print(f"WARNING: sklearn patch failed: {_e}", flush=True)


# === PATCH: Add xpx (sklearn.externals.array_api_extra) shim for sklearn 1.6.1 ===
try:
    import sklearn.utils._array_api as _apx
    if not hasattr(_apx, 'xpx'):
        # Create a minimal xpx shim module with setdiff1d (used by sklearn.utils._encode)
        import types as _xtypes
        _xpx_mod = _xtypes.ModuleType('xpx')
        def _xpx_setdiff1d(x1, x2, /, *, assume_unique=False, xp=None):
            import numpy as _np
            return _np.setdiff1d(_np.asarray(x1).ravel(), _np.asarray(x2).ravel())
        _xpx_mod.setdiff1d = _xpx_setdiff1d
        _apx.xpx = _xpx_mod
        print('Patched sklearn.utils._array_api: xpx shim added', flush=True)
    else:
        print('sklearn.utils._array_api.xpx already present', flush=True)
except Exception as _e:
    print(f"WARNING: xpx patch failed: {_e}", flush=True)

# === PATCH: Fix sklearn.utils.fixes missing _ensure_sparse_index_int32 (sklearn 1.6.1) ===
try:
    import sklearn.utils.fixes as _fixes
    if not hasattr(_fixes, '_ensure_sparse_index_int32'):
        def _ensure_sparse_index_int32_shim(A):
            try:
                from sklearn.utils.fixes import _safely_cast_index_arrays
            except ImportError:
                return
            if hasattr(A, 'format') and A.format in ('csc', 'csr', 'bsr'):
                A.indices, A.indptr = _safely_cast_index_arrays(A)
            elif hasattr(A, 'format') and A.format == 'coo':
                if hasattr(A, 'coords'):
                    A.coords = _safely_cast_index_arrays(A)
                elif hasattr(A, 'indices'):
                    A.indices = _safely_cast_index_arrays(A)
                elif hasattr(A, 'row') and hasattr(A, 'col'):
                    A.row, A.col = _safely_cast_index_arrays(A)
            elif hasattr(A, 'format') and A.format == 'dia':
                A.offsets = _safely_cast_index_arrays(A)
        _fixes._ensure_sparse_index_int32 = _ensure_sparse_index_int32_shim
        print('Patched sklearn.utils.fixes: _ensure_sparse_index_int32 shim added', flush=True)
    else:
        print('sklearn.utils.fixes._ensure_sparse_index_int32 already present', flush=True)
except Exception as _e:
    print(f"WARNING: fixes patch failed: {_e}", flush=True)

# === PATCH: Make is_sklearn_available() return False (sklearn 1.6.1 C extension bugs) ===
# sklearn 1.6.1 on Python 3.12 has multiple C extension symbol mismatches:
#   - sparsefuncs_fast.cpython-312: missing csr_matmul_csr_to_dense
#   - _array_api: missing xpx
#   - fixes: missing _ensure_sparse_index_int32
# These break transformers.generation.candidate_generator -> sklearn.metrics.roc_curve chain.
# Fix: patch transformers.utils.import_utils.is_sklearn_available to return False,
# so the entire sklearn import chain is skipped.
try:
    # Patch transformers.utils.import_utils.is_sklearn_available to return False
    import transformers.utils.import_utils as _iu
    _iu.is_sklearn_available = lambda: False
    print('Patched transformers.utils.import_utils.is_sklearn_available -> False', flush=True)
    
    # Also patch the module-level function in import_utils.py on disk
    _iu_path = '/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py'
    with open(_iu_path) as _f:
        _iu_src = _f.read()
    if 'def is_sklearn_available' in _iu_src:
        _def_idx = _iu_src.index('def is_sklearn_available')
        _next_def = _iu_src.index('def ', _def_idx + 100)
        _old_func = _iu_src[_def_idx:_next_def]
        _new_func = 'def is_sklearn_available() -> bool:\n'
        _new_func += '    """Whether scikit-learn is installed."""\n'
        _new_func += '    return False  # patched: sklearn 1.6.1 C extension bugs on Python 3.12\n'
        _iu_src = _iu_src.replace(_old_func, _new_func)
        with open(_iu_path, 'w') as _f:
            _f.write(_iu_src)
        print('Patched is_sklearn_available() in import_utils.py to return False', flush=True)
    else:
        print('is_sklearn_available not found in import_utils.py', flush=True)
except Exception as _e:
    print(f'WARNING: is_sklearn_available patch failed: {_e}', flush=True)



import kev
print(f"kev imported: {kev.__file__}", flush=True)


In [ ]:
import os, sys, json, torch, kev, kev.model
from pathlib import Path
import time, glob, subprocess

print(f"torch={torch.__version__}", flush=True)

print("Uninstalling torchvision (keep torchaudio)...", flush=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'],
    capture_output=True, text=True
)

try:
    import torchvision
    print(f"WARNING: torchvision still importable: {torchvision.__version__}", flush=True)
except ImportError:
    print("torchvision successfully removed", flush=True)

try:
    from transformers.utils.import_utils import is_torchaudio_available
    print(f"is_torchaudio_available()={is_torchaudio_available()}", flush=True)
except Exception as e:
    print(f"is_torchaudio_available check failed: {e}", flush=True)

HF_MODEL_ID = "Qwen/Qwen2.5-0.5B"
MODEL_PATH = "/tmp/qwen2.5-0.5b"

print(f'Downloading {HF_MODEL_ID} to {MODEL_PATH} via snapshot_download(local_dir=...)', flush=True)
t0 = time.time()

from huggingface_hub import snapshot_download
snapshot_download(
    HF_MODEL_ID,
    local_dir=MODEL_PATH,
    local_dir_use_symlinks=False,
)
print(f'Download complete in {time.time()-t0:.1f}s', flush=True)
print(f'Model path: {MODEL_PATH}', flush=True)
print(f'Path exists: {os.path.isdir(MODEL_PATH)}', flush=True)

print(f'\nContents of {MODEL_PATH}:', flush=True)
for item in sorted(os.listdir(MODEL_PATH)):
    full = os.path.join(MODEL_PATH, item)
    if os.path.isdir(full):
        print(f'  [{item}/]', flush=True)
    else:
        sz = os.path.getsize(full)
        if sz > 1024*1024:
            print(f'  {item} ({sz/(1024*1024):.1f} MB)', flush=True)
        else:
            print(f'  {item} ({sz/1024:.1f} KB)', flush=True)

tok_files = [f for f in os.listdir(MODEL_PATH) if 'tokenizer' in f.lower()]
print(f'\nTokenizer files found: {tok_files}', flush=True)
if not tok_files:
    print("ERROR: No tokenizer files at top level!", flush=True)
    raise RuntimeError("No tokenizer files found in MODEL_PATH")

from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=True,
)
print(f'Tokenizer loaded: {tok.__class__.__name__}', flush=True)

t0 = time.time()
model_raw = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    dtype=torch.float32,
    trust_remote_code=True,
)
print(f'Model loaded in {time.time()-t0:.1f}s', flush=True)

tok = kev.model.load_tokenizer(MODEL_PATH, revision=None)
print(f'Tokenizer: {tok.__class__.__name__}', flush=True)

output_dir = '/tmp/kev-wcag-model'
os.makedirs(output_dir, exist_ok=True)

model = kev.model.DecisionModel(
    MODEL_PATH, tok, 'cuda',
    lora=16,
    head_dim=256,
    dtype=torch.float32,
    revision=None,
)
print(f'Model created successfully', flush=True)
print(f'Trainable params: {sum(p.numel() for p in model.trainable_parameters())/1e6:.1f}M', flush=True)
print(f'Total params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M', flush=True)

max_steps = 15
lr = 2e-4
print(f'Starting training: {max_steps} steps, lr={lr}', flush=True)

t0 = time.time()
model.train()

import torch.nn.functional as F
from kev.model import encode

optimizer = torch.optim.AdamW(model.trainable_parameters(), lr=lr)
global_step = 0
running_loss = 0.0

for step in range(max_steps):
    rec = train_data[global_step % len(train_data)]
    enc = encode(tok, rec, max_state=384, max_branch=1024)
    logits_list = model.forward(enc)

    loss = 0.0
    n_q = 0
    for qi, q in enumerate(rec['questions']):
        if qi < len(logits_list):
            logits = logits_list[qi]
            label = q['label']
            if label < len(logits):
                loss += F.cross_entropy(logits[None], torch.tensor([label], device='cuda'))
                n_q += 1
    if n_q > 0:
        loss = loss / n_q
    else:
        loss = torch.tensor(0.0, device='cuda')

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.trainable_parameters(), 1.0)
    optimizer.step()

    running_loss += loss.item()
    global_step += 1

    if (step + 1) % 5 == 0 or step == 0:
        avg_loss = running_loss / (step + 1)
        elapsed = time.time() - t0
        print(f'  step {step+1}/{max_steps} loss={avg_loss:.4f} ({elapsed:.1f}s)', flush=True)

elapsed = time.time() - t0
print(f'\nTraining complete in {elapsed:.1f}s', flush=True)
print(f'Final loss: {running_loss / max_steps:.4f}', flush=True)

model.lm.save_pretrained(output_dir)
torch.save(model.head.state_dict(), os.path.join(output_dir, 'head.pt'))
tok.save_pretrained(output_dir)
print(f'Checkpoint saved to {output_dir}', flush=True)

ckpt_files = glob.glob(os.path.join(output_dir, '**'), recursive=True)
print(f'Saved {len(ckpt_files)} files:', flush=True)
for cf in sorted(ckpt_files)[:10]:
    sz = os.path.getsize(cf) / 1024
    print(f'  {os.path.relpath(cf, output_dir)}: {sz:.1f} KB', flush=True)
if len(ckpt_files) > 10:
    print(f'  ... and {len(ckpt_files) - 10} more', flush=True)


In [ ]:
import os, glob
from pathlib import Path
output_dir = '/tmp/kev-wcag-model'
print('=== Output directory ===', flush=True)
print(f'Exists: {os.path.isdir(output_dir)}', flush=True)
if os.path.isdir(output_dir):
    for f in sorted(Path(output_dir).glob('**/*')):
        if f.is_file():
            print(f'  {f.relative_to(output_dir)}: {f.stat().st_size/1024:.1f} KB', flush=True)
else:
    print('  (not found)', flush=True)